# Machine learning in the browser

This notebook implements classic machine learning algorithms using
NumPy — entirely in your browser via WebAssembly.

NumPy and Matplotlib are pre-installed in the kernel. At the end we'll
also install an additional package with `%conda install` to show that
conda works at runtime in the browser.

## K-means clustering (pure NumPy)

Implement Lloyd's algorithm from scratch — iterative assignment and centroid
update. No LAPACK or external libraries needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

def make_blobs(n_per_cluster=75, centers=None):
    """Generate clustered 2D data."""
    points = []
    for cx, cy in centers:
        points.append(rng.normal(loc=[cx, cy], scale=0.8, size=(n_per_cluster, 2)))
    return np.vstack(points)

def kmeans(X, k, max_iter=50):
    """Lloyd's algorithm: assign points to nearest centroid, update centroids."""
    idx = rng.choice(len(X), k, replace=False)
    centroids = X[idx].copy()
    for _ in range(max_iter):
        dists = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        labels = np.argmin(dists, axis=1)
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids
    return labels, centroids

centers = [(-3, -3), (0, 3), (3, -1), (-1, 1)]
X = make_blobs(centers=centers)

labels, centroids = kmeans(X, k=4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(X[:, 0], X[:, 1], s=15, color="gray", alpha=0.5)
ax1.set_title("Unlabeled data")

colors = ["#3b82f6", "#ef4444", "#22c55e", "#f59e0b"]
for i in range(4):
    mask = labels == i
    ax2.scatter(X[mask, 0], X[mask, 1], s=15, color=colors[i], alpha=0.6)
ax2.scatter(centroids[:, 0], centroids[:, 1], s=200, c="black", marker="X",
            edgecolors="white", linewidths=2, zorder=5)
ax2.set_title("K-means clustering (k=4)")

for ax in (ax1, ax2):
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("K-means — pure NumPy in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## Binary classification with gradient descent

Train a logistic regression classifier on a 2D dataset and visualize the
decision boundary — all using pure NumPy operations.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def make_moons(n=200):
    """Generate two interleaving half-circles."""
    t = np.linspace(0, np.pi, n // 2)
    x1 = np.column_stack([np.cos(t), np.sin(t)])
    x2 = np.column_stack([1 - np.cos(t), 0.5 - np.sin(t)])
    X = np.vstack([x1, x2]) + rng.normal(scale=0.15, size=(n, 2))
    y = np.concatenate([np.zeros(n // 2), np.ones(n // 2)])
    return X, y

X, y = make_moons(300)

# Normalize features
mu, sigma = X.mean(axis=0), X.std(axis=0)
Xn = (X - mu) / sigma

# Add polynomial features for non-linear boundary: [1, x1, x2, x1², x1·x2, x2²]
Xp = np.column_stack([np.ones(len(Xn)), Xn, Xn**2, Xn[:, 0:1] * Xn[:, 1:2]])

# Gradient descent for logistic regression
w = np.zeros(Xp.shape[1])
lr, n_iter = 0.5, 200
for _ in range(n_iter):
    p = sigmoid(Xp @ w)
    grad = Xp.T @ (p - y) / len(y)
    w -= lr * grad

# Decision boundary
h = 0.05
xx, yy = np.meshgrid(np.arange(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, h),
                      np.arange(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, h))
grid = np.column_stack([xx.ravel(), yy.ravel()])
gn = (grid - mu) / sigma
gp = np.column_stack([np.ones(len(gn)), gn, gn**2, gn[:, 0:1] * gn[:, 1:2]])
Z = (sigmoid(gp @ w) > 0.5).reshape(xx.shape).astype(float)

from matplotlib.colors import ListedColormap
cm_bg = ListedColormap(["#dbeafe", "#fee2e2"])
cm_pts = ListedColormap(["#3b82f6", "#ef4444"])

accuracy = np.mean((sigmoid(Xp @ w) > 0.5) == y)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(X[:, 0], X[:, 1], c=y, cmap=cm_pts, s=15, edgecolors="k", linewidths=0.3)
ax1.set_title("Two moons dataset")

ax2.contourf(xx, yy, Z, cmap=cm_bg, alpha=0.8)
ax2.scatter(X[:, 0], X[:, 1], c=y, cmap=cm_pts, s=15, edgecolors="k", linewidths=0.3)
ax2.set_title(f"Logistic regression ({accuracy:.0%} accuracy)")

for ax in (ax1, ax2):
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Binary classification — pure NumPy in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## K-nearest neighbors (from scratch)

Classify points by majority vote of their k nearest neighbors — computed
with pure NumPy distance calculations.

In [ ]:
def knn_predict(X_train, y_train, X_test, k=5):
    """Predict labels via majority vote of k nearest neighbors."""
    dists = np.linalg.norm(X_test[:, None] - X_train[None, :], axis=2)
    nearest = np.argsort(dists, axis=1)[:, :k]
    votes = y_train[nearest]
    return np.array([np.bincount(row.astype(int)).argmax() for row in votes])

# Split data: 70% train, 30% test
n_train = int(0.7 * len(X))
idx = rng.permutation(len(X))
X_train, X_test = X[idx[:n_train]], X[idx[n_train:]]
y_train, y_test = y[idx[:n_train]], y[idx[n_train:]]

# Compare k values
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, k in zip(axes, [1, 5, 15]):
    y_pred = knn_predict(X_train, y_train, X_test, k=k)
    acc = np.mean(y_pred == y_test)

    grid_pred = knn_predict(X_train, y_train, grid, k=k)
    Z = grid_pred.reshape(xx.shape)

    ax.contourf(xx, yy, Z, cmap=cm_bg, alpha=0.8)
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_pts,
               s=25, edgecolors="k", linewidths=0.5)
    ax.set_title(f"k={k}  ({acc:.0%} accuracy)")
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("K-nearest neighbors — pure NumPy in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## Training dynamics

Visualize how the logistic regression loss decreases during gradient descent —
the optimizer converging to a solution.

In [ ]:
w = np.zeros(Xp.shape[1])
losses = []
for i in range(300):
    p = sigmoid(Xp @ w)
    loss = -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
    losses.append(loss)
    grad = Xp.T @ (p - y) / len(y)
    w -= 0.5 * grad

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(losses, color="steelblue", linewidth=1.5)
ax1.set(xlabel="Iteration", ylabel="Cross-entropy loss",
        title="Training loss over 300 iterations")
ax1.grid(alpha=0.3)

accuracies = []
for k in [1, 3, 5, 7, 11, 15, 21]:
    pred = knn_predict(X_train, y_train, X_test, k=k)
    accuracies.append((k, np.mean(pred == y_test)))

ks, accs = zip(*accuracies)
ax2.plot(ks, accs, "o-", color="tomato", linewidth=1.5, markersize=6)
ax2.set(xlabel="k (neighbors)", ylabel="Test accuracy",
        title="k-NN accuracy vs k")
ax2.set_ylim(0.7, 1.05)
ax2.grid(alpha=0.3)

fig.suptitle("Model evaluation — pure NumPy in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## Install a package at runtime

Want to see the math behind gradient descent? Install
[SymPy](https://www.sympy.org/) — a symbolic math library — right here
in your browser with `%conda install`.

In [ ]:
%load_ext conda_emscripten

In [ ]:
%conda install sympy

In [ ]:
import sympy as sp

# Derive the logistic regression gradient symbolically
z = sp.Symbol("z")
sigma = 1 / (1 + sp.exp(-z))
log_loss = -(sp.Symbol("y") * sp.log(sigma) + (1 - sp.Symbol("y")) * sp.log(1 - sigma))

gradient = sp.simplify(sp.diff(log_loss, z))

print("Logistic regression loss:")
print(f"  L(z) = {log_loss}")
print(f"\nGradient w.r.t. z:")
print(f"  dL/dz = {gradient}")
print(f"\nSimplified:")
print(f"  dL/dz = σ(z) - y")
print(f"\nThis is exactly what our NumPy gradient descent computes above!")

## What just happened

You implemented k-means clustering, logistic regression, and k-nearest
neighbors from scratch, then installed SymPy to derive the gradient
symbolically — all in your browser tab.

Browse available packages at [emscripten-forge.org](https://emscripten-forge.org/)
or use `%conda search <package>` to explore from this notebook.